# Uber Surge Predictor — Experimentation

Explores the dataset, features, and model performance.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import load_data
from src.feature_engineering import create_features

df_raw = load_data()
print(df_raw.shape)
df_raw.head()

## 1. Surge Distribution

In [ ]:
df_raw['surge_multiplier'].hist(bins=40)
plt.title('Surge Multiplier Distribution')
plt.xlabel('Surge'); plt.ylabel('Count')
plt.show()

## 2. Surge by Hour

In [ ]:
df_raw['hour'] = df_raw['timestamp'].dt.hour
df_raw.groupby('hour')['surge_multiplier'].mean().plot(kind='bar', color='steelblue')
plt.title('Average Surge by Hour')
plt.xlabel('Hour'); plt.ylabel('Avg Surge')
plt.tight_layout(); plt.show()

## 3. Feature Engineering

In [ ]:
df = create_features(df_raw.drop(columns=['hour'], errors='ignore'))
print(df.columns.tolist())
df[['lag_1hr_surge','lag_2hr_surge','hour_sin','hour_cos','surge_multiplier']].describe()

## 4. Correlation Heatmap

In [ ]:
num_cols = ['lag_1hr_surge','lag_2hr_surge','hour_sin','hour_cos','surge_multiplier']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation')
plt.show()

## 5. Train Model & Evaluate

In [ ]:
from src.train import train
model = train()

## 6. Feature Importances

In [ ]:
import joblib
pipeline = joblib.load('../models/model.pkl')
rf = pipeline.named_steps['regressor']
preprocessor = pipeline.named_steps['preprocessor']

ohe_cats = preprocessor.named_transformers_['cat'].get_feature_names_out(['geohash'])
feature_names = ['lag_1hr_surge','lag_2hr_surge','hour_sin','hour_cos'] + list(ohe_cats)

importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.plot(kind='bar', color='darkorange')
plt.title('Feature Importances')
plt.tight_layout(); plt.show()

## 7. Time Series Validation (TimeSeriesSplit)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from src.config import FEATURES, TARGET
from src.preprocess import build_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

X = df[FEATURES]; y = df[TARGET]
tscv = TimeSeriesSplit(n_splits=5)

pipe = Pipeline([
    ('pre', build_preprocessor()),
    ('reg', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

scores = cross_val_score(pipe, X, y, cv=tscv, scoring='r2')
print('TimeSeriesSplit R2 scores:', scores.round(4))
print('Mean R2:', scores.mean().round(4))